# Distanza dalla FDT ($I_t$) — ottobre 2026 (solo $h = 10^{-6}$)

Modello di Wilson-Cowan sbilanciato (Nandi et al., *Phys. Rev. E* **107**, 064307 (2023)), $\chi_E = 70\%$, $\chi_I = 30\%$, **solo $h = 10^{-6}$**.

**Convenzione sui dati.** Tutti i campi vengono caricati dal file `_sigmaFIX.npz` e arrotondati a `DECIMALI = 9` cifre decimali (valori sotto $10^{-9}$ = zero).

**Modifiche rispetto a `FDR_integrale.ipynb`**
- Solo $h = 10^{-6}$; rimossi i confronti tra $h$ e la rilettura dei dati grezzi.
- **Mascheratura**: si escludono solo i punti con $W = 0/0$ (primo quadrante, sistema 1D), come per $\beta_{eff}$ in `Wilson_Cowan_ottobre`. Prima si escludevano anche i punti con $W=\infty$.
- **Griglia temporale adattata a ciascun punto** (prima: un'unica griglia con passo $\approx 1.2$ ms, troppo grossolana rispetto a tempi di rilassamento $\sim 1$ ms e periodi di oscillazione $\sim 1$ ms, con errori del 10-25% su $I_t$).
- $y_\infty$ **esatto** ($=\beta_{eff} = -w/[(xw-yz)\sigma_{11}]$) invece della media sull'ultimo 10% della traiettoria.
- **Tre definizioni affiancate** (Sez. 2.3): senza $1/T$, con $1/T$ globale, e normalizzata con il tempo di rilassamento locale $\tilde I = I^{(\infty)}/\tau_{slow}$.
- **Confronto con il tasso di entropia $W$** lungo le linee (Sez. 4), per capire dove $I$ misura davvero la distanza dall'equilibrio.
- Rimossi commenti che rimandavano a sezioni di altri notebook.
- $y_\infty$ calcolato come limite della formula di $\chi(t)$ invece che come $-w/[(xw-yz)\sigma_{11}]$ (piu' preciso vicino al punto critico, Sez. 2.5.1).
- **Esclusa la regione proibita** ($w_{EI}<0$ o $w_{IE}<0$, righe $\delta_{EI} = -7.0, -6.9$), come in `Wilson_Cowan_ottobre`.

## 1. Import, parametri e dati

In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# FILE E PARAMETRI
# ============================================================
# Etichetta del campo esterno h (usata solo nei titoli) e file dei dati.
# Il file .npz contiene, per ogni punto della griglia (delta_EI, delta_IE):
# punto fisso (E0, I0, ...), matrice di drift A in base (E,I) e (Sigma,Delta),
# covarianza sigma (base Sigma,Delta) e autovalori di A.
h_label = r"10^{-6}"
filename = "grid_dynamics_7030_h1e-06_sigmaFIX.npz"

# Parametri del modello di Wilson-Cowan (Nandi et al. 2023)
alpha = 0.1      # tasso di disattivazione dei neuroni (1/ms)
wEE = 6.95       # peso sinaptico E -> E (fisso)
wII = 6.85       # peso sinaptico I -> I (fisso)
chiE = 0.7       # frazione di neuroni eccitatori
chiI = 0.3       # frazione di neuroni inibitori

# Numero di cifre decimali a cui si arrotondano i dati al caricamento:
# valori sotto 1e-9 diventano zero (rumore numerico del solutore).
DECIMALI = 9

# Stile dei grafici "da articolo"
ARTICLE_LAYOUT = dict(
    font=dict(family="Times New Roman, Georgia, serif", size=14, color="black"),
    title_font=dict(family="Times New Roman, Georgia, serif", size=18),
)

# ============================================================
# SALVATAGGIO AUTOMATICO DELLE FIGURE (richiede 'kaleido')
# ============================================================
CARTELLA_FIGURE = "figure_esportate_FDR_ottobre"
os.makedirs(CARTELLA_FIGURE, exist_ok=True)


def salva_figura(fig, nome_file, scale=2):
    """Salva la figura come PNG (CARTELLA_FIGURE/nome_file.png) e poi la mostra."""
    path = os.path.join(CARTELLA_FIGURE, f"{nome_file}.png")
    try:
        fig.write_image(path, scale=scale)
        print(f"Figura salvata: {path}")
    except Exception as e:
        print(f"ATTENZIONE: impossibile salvare '{path}' ({e}). "
              f"Installa il pacchetto con: pip install kaleido")
    fig.show()


def _trapz(y, x):
    """np.trapezoid (numpy >= 2.0) oppure np.trapz (numpy < 2.0)."""
    # Integrale numerico con la regola dei trapezi; il nome della funzione
    # numpy e' cambiato tra le versioni, quindi si usa quella disponibile.
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)


# ============================================================
# CARICAMENTO DATI (arrotondati a DECIMALI cifre, assi esclusi)
# ============================================================
# Gli array 2D hanno indice 0 = delta_EI (righe), indice 1 = delta_IE (colonne).
# Gli assi della griglia non vengono arrotondati.
raw = np.load(filename)
data = {}
for key in raw.files:
    if key in ("delta_EI", "delta_IE"):
        data[key] = raw[key]
    else:
        data[key] = np.round(raw[key], DECIMALI)

# ============================================================
# REGIONE PROIBITA: w_EI < 0 oppure w_IE < 0
# ============================================================
# I pesi sinaptici sono intensita' e devono essere positivi
# (w_EI = w_II + delta_EI, w_IE = w_EE + delta_IE). Sulla griglia attuale
# sono le righe delta_EI = -7.0 e -6.9. Questi punti vengono esclusi da
# tutte le analisi (W messo a NaN, I_punto li salta).
DEI_griglia, DIE_griglia = np.meshgrid(data["delta_EI"], data["delta_IE"], indexing="ij")
proibita = ((wII + DEI_griglia) < 0) | ((wEE + DIE_griglia) < 0)

print(f"h = {h_label}  <-  {filename}")
print(f"griglia: {len(data['delta_EI'])} x {len(data['delta_IE'])} (delta_EI x delta_IE)")
print(f"punti nella regione proibita (esclusi): {int(proibita.sum())}, "
      f"righe delta_EI = {np.round(data['delta_EI'][np.any(proibita, axis=1)], 2)}")

h = 10^{-6}  <-  grid_dynamics_7030_h1e-06_sigmaFIX.npz
griglia: 141 x 140 (delta_EI x delta_IE)
punti nella regione proibita (esclusi): 280, righe delta_EI = [-7.  -6.9]


### 1.1 Tasso di entropia e maschera $W = 0/0$

$W = -\delta^2 / [(A_{EE}+A_{II})E_0 I_0]$, con $\delta = \sqrt{\chi_E/\chi_I}A_{EI}I_0 - \sqrt{\chi_I/\chi_E}A_{IE}E_0$. Oltre alla regione proibita, si escludono **solo** i punti dove numeratore e denominatore sono entrambi nulli ($E_0 = 0$ e $A_{EI} = 0$: primo quadrante, sistema 1D, equilibrio). I punti con $W = \infty$ ($E_0 = 0$ ma $A_{EI}\neq 0$, sistema ancora 2D) vengono analizzati normalmente.

In [2]:
# ============================================================
# TASSO DI PRODUZIONE DI ENTROPIA W E MASCHERA W = 0/0
# ============================================================
# W = -delta^2 / [Tr(A) E0 I0], con
# delta = sqrt(chiE/chiI) A_EI I0 - sqrt(chiI/chiE) A_IE E0.
E0 = data["E0"]; I0 = data["I0"]
delta_term = (np.sqrt(chiE / chiI) * data["A_EI"] * I0
              - np.sqrt(chiI / chiE) * data["A_IE"] * E0)
W_num = delta_term**2                                   # numeratore delta^2
W_den = (data["A_EE"] + data["A_II"]) * E0 * I0         # denominatore Tr(A) E0 I0

# Divisioni per zero ammesse: producono inf (num != 0, den = 0) o nan (0/0)
with np.errstate(divide="ignore", invalid="ignore"):
    W = -W_num / W_den

# Regione proibita: W non definito (NaN)
W = np.where(proibita, np.nan, W)

# Punti 0/0: E0 = 0 e A_EI = 0 (primo quadrante, sistema 1D all'equilibrio).
# E' l'unica maschera usata nel notebook.
W_00 = (W_num == 0) & (W_den == 0) & ~proibita     # maschera dei punti 1D (oltre alla regione proibita)

# NOTA (da discutere): l'unico punto con W = inf (delta_EI = 0.1,
# delta_IE = 0) e' risultato essere un artefatto numerico (E0 ~ 3.6e-12
# arrotondato a zero, S_E ~ 1e-16), vedi Wilson_Cowan_ottobre, Sez. 10.
# Qui viene analizzato come un punto normale.

print(f"regione proibita (esclusi)  : {int(proibita.sum())}")
print(f"W = 0/0 (mascherati)        : {int(W_00.sum())}")
print(f"W = inf (NON mascherati)    : {int(np.isinf(W).sum())}")
print(f"W finito                    : {int(np.isfinite(W).sum())}")

regione proibita (esclusi)  : 280
W = 0/0 (mascherati)        : 6544
W = inf (NON mascherati)    : 1
W finito                    : 12915


---
## 2. Distanza dalla FDT

**Definizioni**
$$x(t) = \frac{C_{\Sigma\Sigma}(t)}{C_{\Sigma\Sigma}(0)}, \qquad y(t) = \frac{\chi_{\Sigma\Sigma}(t)}{C_{\Sigma\Sigma}(0)}, \qquad \chi_{\Sigma\Sigma}(t) = \int_0^t R_{\Sigma\Sigma}(t')\,dt' .$$

Se la FDT vale, $\chi(t) = \beta\,[C(0)-C(t)]$, cioe' $y(t)/y_\infty = 1 - x(t)$. L'integrando che misura la violazione e'
$$\Delta(t) = \left|\frac{y(t)}{y_\infty} - \bigl(1-x(t)\bigr)\right| .$$

$y_\infty$ e' calcolato **esattamente**: $\int_0^\infty e^{At}dt = -A^{-1}$, quindi $y_\infty = -(A^{-1})_{11}/\sigma_{11} = -w/[(xw-yz)\,\sigma_{11}] = \beta_{eff}$.

$C_{\Sigma\Sigma}(t)$ (Eq. 10, 14) e $\chi_{\Sigma\Sigma}(t)$ (integrale di Eq. A38, A39) sono in forma chiusa.

### 2.1 Griglia temporale adattata

Ogni punto ha la sua griglia, costruita sui propri tempi caratteristici:

- **Autovalori complessi** ($\lambda = a \pm ib$): un solo tempo di decadimento $\tau = 1/|a|$ e un periodo $2\pi/b$. Passo $dt = \min(\tau, 2\pi/b)/\text{PPS}$, fino a $T_{loc} = K\tau$.
- **Autovalori reali** ($\lambda_1 > \lambda_2$): due scale, $\tau_{lento} = 1/|\lambda_1|$ e $\tau_{veloce} = 1/|\lambda_2|$. Primo tratto $[0, K\tau_{veloce}]$ con passo $\tau_{veloce}/\text{PPS}$, secondo tratto fino a $T_{loc} = K\tau_{lento}$ con passo $\tau_{lento}/\text{PPS}$.

Con $K = 21$, a $T_{loc}$ l'integrando e' sceso sotto $e^{-21} \approx 10^{-9}$ (la precisione dei dati), quindi $\int_0^{T_{loc}} = \int_0^\infty$ alla precisione usata. `PPS` = punti per tempo caratteristico.

In [3]:
# Parametri della griglia temporale
K_TAU = 21          # tempi caratteristici inclusi
PPS = 50            # punti per tempo caratteristico (o per periodo)
N_MAX = 4_000_000   # tetto di sicurezza sul numero di punti per segmento


def griglia_adattata(is_cplx, a, b, L1, L2):
    """Griglia temporale per un punto. Ritorna (t, T_loc, tau_slow)."""
    # --- Autovalori complessi lambda = a +- i b ---
    # Un solo tempo di decadimento tau = 1/|a| e un periodo 2*pi/b:
    # il passo e' il piu' piccolo dei due diviso PPS, fino a T_loc = K_TAU * tau.
    if is_cplx == 1:
        tau = -1.0 / a
        scala = min(tau, 2 * np.pi / b)
        T_loc = K_TAU * tau
        n = int(min(np.ceil(T_loc / (scala / PPS)) + 1, N_MAX))
        return np.linspace(0.0, T_loc, n), T_loc, tau

    # --- Autovalori reali lambda1 > lambda2 (entrambi negativi) ---
    # Due scale: tau_s = 1/|lambda1| (lenta) e tau_f = 1/|lambda2| (veloce).
    tau_s = -1.0 / L1
    tau_f = -1.0 / L2
    T_loc = K_TAU * tau_s
    # Primo tratto [0, K_TAU*tau_f], fitto (passo tau_f/PPS): risolve il modo veloce
    t1 = min(T_loc, K_TAU * tau_f)
    n1 = int(min(np.ceil(t1 / (tau_f / PPS)) + 1, N_MAX))
    seg1 = np.linspace(0.0, t1, n1)
    if T_loc <= t1:
        return seg1, T_loc, tau_s
    # Secondo tratto [t1, T_loc], piu' rado (passo tau_s/PPS): resta solo il modo lento.
    # [1:] evita di ripetere il punto t1 gia' presente nel primo tratto.
    n2 = int(min(np.ceil((T_loc - t1) / (tau_s / PPS)) + 1, N_MAX))
    seg2 = np.linspace(t1, T_loc, n2)[1:]
    return np.concatenate([seg1, seg2]), T_loc, tau_s

### 2.2 Funzioni core

In [4]:
def x_y_punto(i, j, t):
    """x(t) = C(t)/C(0) e y(t) = chi(t)/C(0) nel punto (i, j), piu' y_inf esatto."""
    # Elementi della matrice di drift in base (Sigma, Delta): A = [[X, Y], [Z, Wm]]
    X = data["A_x"][i, j]; Y = data["A_y"][i, j]; Z = data["A_z"][i, j]; Wm = data["A_w"][i, j]
    # Covarianza stazionaria: s11 = C_SigmaSigma(0), s12 = C_SigmaDelta(0)
    s11 = data["sigma11"][i, j]; s12 = data["sigma12"][i, j]

    if data["is_complex"][i, j] == 1:
        # Autovalori complessi lambda = a +- i b:
        # C(t) = autocorrelazione di Sigma (Eq. 14 di Nandi et al.)
        # chi(t) = integrale da 0 a t di R_SigmaSigma (forma chiusa di Eq. A39)
        a = data["lambda_re"][i, j]; b = data["lambda_im"][i, j]
        ea = np.exp(a * t); cb = np.cos(b * t); sb = np.sin(b * t)
        C = s11 * ea * cb + ((X - a) * s11 + Y * s12) / b * ea * sb
        chi = ((2 * a - X) * ea * cb + (b**2 + X * a - a**2) / b * ea * sb + (X - 2 * a)) / (a**2 + b**2)
        # chi(inf): limite t -> inf della stessa formula (i termini con e^{at} vanno a zero)
        chi_inf = (X - 2 * a) / (a**2 + b**2)
    else:
        # Autovalori reali lambda1 > lambda2:
        # C(t) (Eq. 10) e chi(t) = integrale di R_SigmaSigma (forma chiusa di Eq. A38)
        L1 = data["lambda1"][i, j]; L2 = data["lambda2"][i, j]
        rad = L1 - L2                       # = sqrt((x-w)^2 + 4yz)
        e1 = np.exp(L1 * t); e2 = np.exp(L2 * t)
        C = (((X - L2) * s11 + Y * s12) * e1 - ((X - L1) * s11 + Y * s12) * e2) / rad
        chi = (L2 * (X - L2) * (e1 - 1) - L1 * (X - L1) * (e2 - 1)) / (L1 * L2 * rad)
        # chi(inf): limite t -> inf della stessa formula (e1, e2 -> 0)
        chi_inf = (L1 * (X - L1) - L2 * (X - L2)) / (L1 * L2 * rad)

    # y_inf = chi(inf)/C(0) = beta_eff.
    # Matematicamente chi_inf = -(A^-1)_11 = -w/(xw - yz), ma si usa il limite della
    # stessa formula di chi(t): cosi' y(t) -> y_inf esattamente anche con i dati arrotondati.
    # La formula -w/(xw - yz) vicino al punto critico (det A -> 0) e' una differenza di
    # numeri quasi uguali (xw ~ yz ~ -125) e amplifica l'arrotondamento a 9 cifre:
    # al punto critico dava |y(fine)/y_inf - 1| ~ 7e-5 e un integrando che non andava a zero
    # (vedi Sez. 2.5.1).
    y_inf = chi_inf / s11
    return C / s11, chi / s11, y_inf


def I_punto(i, j):
    """Integrale di Delta(t) su [0, T_loc] con griglia adattata."""
    # Punti esclusi: regione proibita, primo quadrante (1D) e covarianza nulla
    if proibita[i, j]:
        return dict(I=np.nan, reason="regione proibita")
    if W_00[i, j]:
        return dict(I=np.nan, reason="W_0/0 (1D, mascherato)")
    if data["sigma11"][i, j] == 0:
        return dict(I=np.nan, reason="sigma11 = 0")

    # Autovalori: devono essere stabili (parte reale < 0) e non degeneri
    is_c = data["is_complex"][i, j]
    a, b = data["lambda_re"][i, j], data["lambda_im"][i, j]
    L1, L2 = data["lambda1"][i, j], data["lambda2"][i, j]
    if (is_c == 1 and (a >= 0 or b == 0)) or (is_c == 0 and (L1 >= 0 or L1 == L2)):
        return dict(I=np.nan, reason="autovalori degeneri o non stabili")

    t, T_loc, tau_slow = griglia_adattata(is_c, a, b, L1, L2)
    x_v, y_v, y_inf = x_y_punto(i, j, t)
    if not np.isfinite(y_inf) or y_inf == 0:
        return dict(I=np.nan, reason="y_inf = 0 o non finito")

    # Integrando: distanza tra la risposta normalizzata y/y_inf e la previsione FDT 1 - x
    # NOTA (da discutere): vicino agli zeri di w = A_ww si ha y_inf -> 0 e
    # I -> 1/|w| (verificato sui dati): i picchi li' vengono dalla normalizzazione.
    integrando = np.abs(y_v / y_inf - (1.0 - x_v))
    # Controllo di decadimento: a T_loc devono valere x ~ 0 e y ~ y_inf
    fine_ok = (abs(x_v[-1]) < 1e-5) and (abs(y_v[-1] / y_inf - 1) < 1e-5)
    return dict(I=_trapz(integrando, t), reason="ok", T_loc=T_loc, tau_slow=tau_slow,
                n_punti=len(t), y_inf=y_inf, decaduto=fine_ok)


def I_lungo_linea(axis, target):
    """axis='dei': delta_EI fissato (varia delta_IE); axis='die': delta_IE fissato."""
    # Si prende la riga (o colonna) della griglia piu' vicina al valore richiesto
    if axis == "dei":
        i0 = int(np.argmin(np.abs(data["delta_EI"] - target)))
        punti = [(i0, j) for j in range(len(data["delta_IE"]))]
        var_vals = data["delta_IE"]
    else:
        j0 = int(np.argmin(np.abs(data["delta_IE"] - target)))
        punti = [(i, j0) for i in range(len(data["delta_EI"]))]
        var_vals = data["delta_EI"]

    # Calcolo punto per punto e raccolta dei risultati in array
    out = [I_punto(i, j) for (i, j) in punti]
    return dict(
        var_vals=np.asarray(var_vals),
        punti=punti,
        I=np.array([o["I"] for o in out]),
        reason=np.array([o["reason"] for o in out], dtype=object),
        tau_slow=np.array([o.get("tau_slow", np.nan) for o in out]),
        n_punti=np.array([o.get("n_punti", 0) for o in out]),
        y_inf=np.array([o.get("y_inf", np.nan) for o in out]),
        decaduto=np.array([o.get("decaduto", False) for o in out]),
    )

### 2.3 Con o senza $1/T$?

Le due definizioni sono
$$I^{(\infty)} = \int_0^{\infty} \Delta(t)\,dt \qquad\text{e}\qquad I_T = \frac{1}{T}\int_0^{T} \Delta(t)\,dt .$$

$\Delta(t)$ **decade a zero** in qualche $\tau_{slow}$ (perche' $x\to 0$ e $y\to y_\infty$). Quindi, appena $T$ e' molto piu' grande di $\tau_{slow}$, l'integrale smette di crescere e
$$I_T \simeq \frac{I^{(\infty)}}{T} .$$
$I_T$ e' cioe' $I^{(\infty)}$ diviso per un numero che non dipende dal punto: la **forma** delle curve (posizione dei picchi, rapporti tra punti) e' identica, cambia solo la scala verticale. Il problema e' che quella scala dipende dalla scelta di $T$: con il $T$ del vecchio notebook (fissato dal punto critico a $h = 10^{-9}$) i valori erano circa 7 volte piu' piccoli di quelli che si ottengono con il $T$ calcolato solo su $h = 10^{-6}$, senza che sia cambiata la fisica.

- **Senza $1/T$** ($I^{(\infty)}$): quantita' ben definita, indipendente da scelte numeriche, con le dimensioni di un tempo (ms). E' la scelta consigliata per valori assoluti.
- **Con $1/T$** ($I_T$): ha senso come *media temporale* della violazione su una finestra fissata da motivi fisici o sperimentali (es. la durata di una registrazione). Va bene per confronti tra punti **a parita' di $T$**, ma il valore assoluto non ha significato intrinseco.

**Terza opzione: $\tilde I = I^{(\infty)}/\tau_{slow}$ (tempo locale).** $I^{(\infty)}$ ha le dimensioni di un tempo, quindi cresce con la lentezza del punto: due punti con la stessa violazione "di forma", uno che rilassa in 1 ms e uno in 10 ms, hanno $I^{(\infty)}$ in rapporto 1:10. Dividendo per il $\tau_{slow}$ **del punto stesso** si ottiene una quantita' adimensionale, la violazione media per tempo di rilassamento, che non premia i punti lenti. A differenza di $1/T$ globale, questa normalizzazione cambia da punto a punto e quindi **puo' cambiare la forma delle curve e la posizione dei massimi**.

Qui $T$ e' scelto come prima, ma solo su $h=10^{-6}$: $T = \mathrm{clip}(K\,\tau_{slow}^{max}, 1000, 250000)$, con $\tau_{slow}^{max}$ massimo sulle linee analizzate. Poiche' $T \ge T_{loc}$ per ogni punto e oltre $T_{loc}$ l'integrando e' $< 10^{-9}$, $\int_0^T \Delta\,dt = I^{(\infty)}$ alla precisione usata, quindi $I_T = I^{(\infty)}/T$.

In [5]:
# Linee analizzate (stesse della Sez. 6.3 di Wilson_Cowan_ottobre)
valori_dei = [-5, -2.5, -0.5, 0, 2.5, 5]   # linee orizzontali: delta_EI fissato
valori_die = [-5, -2.5, -0.5, 0, 2.5, 5]   # linee verticali: delta_IE fissato

# Limiti inferiore e superiore per il T globale (ms)
T_FLOOR, T_CAP = 1000.0, 250000.0

# Autovalore piu' lento in ogni punto: lambda_re (complessi) o max(lambda1, lambda2) (reali)
lam_slow = np.where(data["is_complex"] == 1, data["lambda_re"], np.maximum(data["lambda1"], data["lambda2"]))
lam_slow = np.where(np.isfinite(lam_slow) & (lam_slow < 0), lam_slow, np.nan)
tau_full = -1.0 / lam_slow           # tau_slow su tutta la griglia

# Maschera dei punti che stanno su una delle linee
mask_linee = np.zeros(tau_full.shape, dtype=bool)
for target in valori_dei:
    mask_linee[int(np.argmin(np.abs(data["delta_EI"] - target))), :] = True
for target in valori_die:
    mask_linee[:, int(np.argmin(np.abs(data["delta_IE"] - target)))] = True

# tau_slow massimo sulle linee (esclusi i punti W = 0/0 e la regione proibita)
tau_lin = np.where(mask_linee & ~W_00 & ~proibita, tau_full, np.nan)
k_max = int(np.nanargmax(tau_lin))
i_max, j_max = np.unravel_index(k_max, tau_lin.shape)
tau_slow_max = float(tau_lin[i_max, j_max])

# T globale = K_TAU * tau_slow_max, limitato a [T_FLOOR, T_CAP]
T_glob = float(np.clip(K_TAU * tau_slow_max, T_FLOOR, T_CAP))
print(f"tau_slow massimo sulle linee: {tau_slow_max:.3f} ms in "
      f"(delta_EI={data['delta_EI'][i_max]:.2f}, delta_IE={data['delta_IE'][j_max]:.2f})")
print(f"T usato per I_T = {T_glob:.0f} ms"
      + ("  <-- CAPPATO" if K_TAU * tau_slow_max > T_CAP else ""))

tau_slow massimo sulle linee: 1581.078 ms in (delta_EI=0.00, delta_IE=0.00)
T usato per I_T = 33203 ms


### 2.4 Calcolo lungo le linee

In [6]:
# Calcolo di I^(inf) su tutte le linee
results_dei = {target: I_lungo_linea("dei", target) for target in valori_dei}
results_die = {target: I_lungo_linea("die", target) for target in valori_die}

# Le altre due definizioni si ottengono da I^(inf):
#   I_T = I^(inf) / T_glob   (stesso T per tutti i punti)
#   I_tilde = I^(inf) / tau_slow   (tempo di rilassamento del punto)
for res in list(results_dei.values()) + list(results_die.values()):
    res["I_T"] = res["I"] / T_glob
    res["I_tilde"] = res["I"] / res["tau_slow"]

def riassunto(results, nome):
    # Conta i punti per motivo di esclusione, i punti decaduti a T_loc
    # e il numero di istanti temporali usati
    tutte = np.concatenate([r["reason"] for r in results.values()])
    dec = np.concatenate([r["decaduto"][r["reason"] == "ok"] for r in results.values()])
    npt = np.concatenate([r["n_punti"][r["reason"] == "ok"] for r in results.values()])
    print(f"--- {nome} ---")
    for motivo, n in zip(*np.unique(tutte, return_counts=True)):
        print(f"  {motivo:35s}: {n}")
    print(f"  punti 'ok' con decadimento completo a T_loc: {int(dec.sum())}/{len(dec)}")
    print(f"  punti temporali per punto: min {npt.min()}, mediana {int(np.median(npt))}, max {npt.max()}")

# Nella versione precedente un punto per famiglia (il punto critico (0, 0)) non
# superava il controllo: il problema era y_inf, non il decadimento (vedi Sez. 2.5.1).
riassunto(results_dei, "linee a delta_EI fissato")
riassunto(results_die, "linee a delta_IE fissato")

--- linee a delta_EI fissato ---
  W_0/0 (1D, mascherato)             : 190
  ok                                 : 650
  punti 'ok' con decadimento completo a T_loc: 650/650
  punti temporali per punto: min 1051, mediana 1836, max 23145
--- linee a delta_IE fissato ---
  W_0/0 (1D, mascherato)             : 306
  ok                                 : 528
  regione proibita                   : 12
  punti 'ok' con decadimento completo a T_loc: 528/528
  punti temporali per punto: min 1051, mediana 1731, max 19561


### 2.5 Controllo della griglia temporale

Nel punto scelto, confronto tra la griglia adattata e una griglia uniforme come quella del vecchio notebook (200000 punti su $[0,T]$), per due valori di $T$: quello calcolato qui e quello vecchio (237971 ms, passo $\approx 1.2$ ms).

In [7]:
# Punto su cui si controlla la griglia temporale
punto_controllo = (2.5, -2.2)    # (delta_EI, delta_IE)

i = int(np.argmin(np.abs(data["delta_EI"] - punto_controllo[0])))
j = int(np.argmin(np.abs(data["delta_IE"] - punto_controllo[1])))
ref = I_punto(i, j)    # valore di riferimento con la griglia adattata
print(f"Punto (delta_EI={data['delta_EI'][i]:.2f}, delta_IE={data['delta_IE'][j]:.2f}), "
      f"tau_slow={ref.get('tau_slow', np.nan):.3f} ms")
print(f"  griglia adattata ({ref.get('n_punti', 0)} punti): I_inf = {ref['I']:.6f}")

if ref["reason"] == "ok":
    # Confronto con griglie uniformi di 200000 punti: con il T attuale e con il
    # T del vecchio notebook (237971 ms, passo ~1.2 ms, troppo grossolano)
    for T_unif in [T_glob, 237971.0]:
        t_u = np.linspace(0, T_unif, 200000)
        x_u, y_u, yi = x_y_punto(i, j, t_u)
        I_u = _trapz(np.abs(y_u / yi - (1 - x_u)), t_u)
        print(f"  griglia uniforme, T={T_unif:.0f}, dt={t_u[1]:.3f} ms: I_inf = {I_u:.6f} "
              f"(errore {100 * (I_u / ref['I'] - 1):+.1f}%)")

    # raddoppio della risoluzione: il risultato non deve cambiare
    # (PPS e' una variabile globale letta da griglia_adattata: la si raddoppia
    # temporaneamente e poi la si rimette al valore originale)
    PPS_salvato = PPS
    PPS = 2 * PPS_salvato
    I_fine = I_punto(i, j)["I"]
    PPS = PPS_salvato
    print(f"  griglia adattata con PPS raddoppiato: I_inf = {I_fine:.6f} "
          f"(differenza relativa {abs(I_fine / ref['I'] - 1):.1e})")

Punto (delta_EI=2.50, delta_IE=-2.20), tau_slow=1.414 ms
  griglia adattata (1773 punti): I_inf = 4.350404
  griglia uniforme, T=33203, dt=0.166 ms: I_inf = 4.334590 (errore -0.4%)
  griglia uniforme, T=237971, dt=1.190 ms: I_inf = 3.650625 (errore -16.1%)
  griglia adattata con PPS raddoppiato: I_inf = 4.350437 (differenza relativa 7.8e-06)


### 2.5.1 Controllo: le due formule per $y_\infty$

$y_\infty = \chi(\infty)/\sigma_{11}$ si puo' calcolare in due modi, matematicamente identici:
- **dagli elementi di $A$:** $\chi(\infty) = -(A^{-1})_{11} = -w/(xw - yz)$;
- **come limite della formula di $\chi(t)$** (quella usata ora in `x_y_punto`): $\chi(\infty) = \dfrac{\lambda_1(x-\lambda_1) - \lambda_2(x-\lambda_2)}{\lambda_1\lambda_2(\lambda_1-\lambda_2)}$ (reali), $\chi(\infty) = \dfrac{x - 2a}{a^2+b^2}$ (complessi).

Usando $\lambda_1+\lambda_2 = x+w$ e $\lambda_1\lambda_2 = xw-yz$ si verifica che coincidono. Ma i dati sono arrotondati a 9 cifre, e vicino al punto critico $\det A = xw - yz \to 0$ e' la differenza di due numeri quasi uguali: l'arrotondamento pesa molto e le due strade danno valori leggermente diversi. Se $y_\infty$ non e' esattamente il valore finale di $y(t)$, l'integrando $\Delta(t) \to |y_\infty^{vero}/y_\infty - 1| \neq 0$ e l'integrale accumula un contributo spurio fino a $T_{loc}$.

La cella confronta, sui punti delle linee, $I^{(\infty)}$ calcolato con il $y_\infty$ attuale e con quello vecchio ($-w/[(xw-yz)\sigma_{11}]$), e mostra i punti dove la differenza e' maggiore.

In [8]:
# ============================================================
# 2.5.1 CONFRONTO: y_inf dagli elementi di A (vecchio) vs limite della formula (attuale)
# ============================================================

def I_con_y_inf_vecchio(i, j):
    """I^(inf) con y_inf = -w / [(xw - yz) sigma11], a parita' di tutto il resto."""
    is_c = data["is_complex"][i, j]
    t, _, _ = griglia_adattata(is_c, data["lambda_re"][i, j], data["lambda_im"][i, j],
                               data["lambda1"][i, j], data["lambda2"][i, j])
    x_v, y_v, _ = x_y_punto(i, j, t)
    X = data["A_x"][i, j]; Y = data["A_y"][i, j]; Z = data["A_z"][i, j]; Wm = data["A_w"][i, j]
    y_inf_vecchio = -Wm / ((X * Wm - Y * Z) * data["sigma11"][i, j])
    return _trapz(np.abs(y_v / y_inf_vecchio - (1.0 - x_v)), t), y_inf_vecchio


righe = []
for nome, results in [("delta_EI", results_dei), ("delta_IE", results_die)]:
    for target, res in results.items():
        for k, (i, j) in enumerate(res["punti"]):
            if res["reason"][k] != "ok":
                continue
            I_old, y_old = I_con_y_inf_vecchio(i, j)
            righe.append({"delta_EI": float(data["delta_EI"][i]), "delta_IE": float(data["delta_IE"][j]),
                          "tau_slow (ms)": res["tau_slow"][k],
                          "y_inf attuale": res["y_inf"][k], "y_inf vecchio": y_old,
                          "diff. rel. y_inf": abs(y_old / res["y_inf"][k] - 1),
                          "I_inf attuale": res["I"][k], "I_inf vecchio": I_old,
                          "diff. rel. I_inf": abs(I_old / res["I"][k] - 1),
                          "decaduto": res["decaduto"][k]})

df_y = pd.DataFrame(righe).drop_duplicates(subset=["delta_EI", "delta_IE"])
print(f"Punti confrontati: {len(df_y)}")
print(f"Differenza relativa su y_inf:  mediana {df_y['diff. rel. y_inf'].median():.1e}, "
      f"max {df_y['diff. rel. y_inf'].max():.1e}")
print(f"Differenza relativa su I_inf:  mediana {df_y['diff. rel. I_inf'].median():.1e}, "
      f"max {df_y['diff. rel. I_inf'].max():.1e}")
print(f"Punti che ora superano il controllo di decadimento: {int(df_y['decaduto'].sum())}/{len(df_y)}")
print("\nI 10 punti con la differenza maggiore su I_inf:")
display(df_y.sort_values("diff. rel. I_inf", ascending=False).head(10))

Punti confrontati: 1151
Differenza relativa su y_inf:  mediana 1.2e-10, max 7.1e-05
Differenza relativa su I_inf:  mediana 1.2e-08, max 2.5e+01
Punti che ora superano il controllo di decadimento: 1151/1151

I 10 punti con la differenza maggiore su I_inf:


,delta_EI,delta_IE,tau_slow (ms),y_inf attuale,y_inf vecchio,diff. rel. y_inf,I_inf attuale,I_inf vecchio,diff. rel. I_inf,decaduto
489,0.0,0.0,1581.077663,-28.286979,-28.288999,7.143359e-05,0.090179,2.348137,2.503854e+01,True
610,2.5,-1.9,3.514648,-4.811968,-4.811968,4.772757e-09,0.177142,0.177142,1.865445e-06,True
894,3.7,-2.5,3.154158,-6.394619,-6.394619,5.694507e-09,0.205401,0.205400,1.107829e-06,True
1039,0.1,0.0,20.000194,-876715.678911,-876715.685757,7.809399e-09,0.931122,0.931123,7.761332e-07,True
895,3.8,-2.5,5.094412,-5.304297,-5.304297,1.148963e-09,0.105207,0.105207,7.564374e-07,True
893,3.6,-2.5,2.461498,-6.443839,-6.443839,3.732499e-09,0.321429,0.321429,4.240361e-07,True
355,-0.5,0.6,20.000296,-43251.550732,-43251.550891,3.666549e-09,0.726410,0.726411,4.182176e-07,True
491,0.0,0.2,20.000193,-502247.695024,-502247.698239,6.403063e-09,1.337662,1.337662,3.873602e-07,True
892,3.5,-2.5,2.082155,-5.850812,-5.850812,3.251326e-09,0.468503,0.468504,2.830463e-07,True
609,2.5,-2.0,2.043897,-5.348783,-5.348783,2.786059e-09,0.469847,0.469847,2.366013e-07,True


### 2.6 Grafici: $I^{(\infty)}$ lungo le linee

$I^{(\infty)} = \int_0^\infty \Delta(t)\,dt$ (in ms, scala log) lungo le 6 linee a $\delta_{EI}$ fissato e lungo le 6 a $\delta_{IE}$ fissato. Le altre due definizioni ($I_T$ e $\tilde I$) sono in Appendice.

In [9]:
# ============================================================
# 2.6 I^(inf) LUNGO LE LINEE
# ============================================================
# Colori e simboli delle 6 linee
PLOTLY_COLORS = [
    "rgb(31,119,180)", "rgb(174,199,232)", "rgb(214,39,40)",
    "rgb(44,160,44)", "rgb(255,127,14)", "rgb(148,103,189)",
]
PLOTLY_SYMBOLS = ["circle", "square", "triangle-up", "diamond", "pentagon", "star"]


def plot_I_inf(results, target_values, xlabel, legend_prefix, title, nome_file):
    # Un solo pannello: una curva di I^(inf) per ciascuna linea
    fig = go.Figure()
    for k, target in enumerate(target_values):
        res = results[target]
        order = np.argsort(res["var_vals"])                         # ordina i punti lungo la linea
        xv = res["var_vals"][order]
        yv = res["I"][order]
        yv = np.where(np.isfinite(yv) & (yv > 0), yv, np.nan)       # scala log: solo valori > 0
        color = PLOTLY_COLORS[k % len(PLOTLY_COLORS)]
        symbol = PLOTLY_SYMBOLS[k % len(PLOTLY_SYMBOLS)]
        fig.add_trace(go.Scatter(
            x=xv, y=yv, mode="lines+markers",
            name=f"{legend_prefix} = {target:g}",
            line=dict(color=color, dash="dash", width=1.5),
            marker=dict(symbol=symbol, size=6, color=color),
            hovertemplate=(f"{legend_prefix} = {target:g}<br>{xlabel} = %{{x:.4g}}<br>"
                           "I_inf = %{y:.6g} ms<extra></extra>"),
        ))
    fig.update_xaxes(title_text=xlabel)
    fig.update_yaxes(title_text="I<sup>(∞)</sup> (ms)", type="log")
    fig.update_layout(title=title, height=550, width=900, hovermode="closest",
                      legend=dict(title=legend_prefix), **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_I_inf(results_dei, valori_dei, "δ<sub>IE</sub>", "δ<sub>EI</sub>",
           f"Distanza dalla FDT I<sup>(∞)</sup>, δ<sub>EI</sub> fissato — h = {h_label}",
           "I_inf_linee_delta_EI_fissato")
plot_I_inf(results_die, valori_die, "δ<sub>EI</sub>", "δ<sub>IE</sub>",
           f"Distanza dalla FDT I<sup>(∞)</sup>, δ<sub>IE</sub> fissato — h = {h_label}",
           "I_inf_linee_delta_IE_fissato")

Figura salvata: figure_esportate_FDR_ottobre\I_inf_linee_delta_EI_fissato.png


Figura salvata: figure_esportate_FDR_ottobre\I_inf_linee_delta_IE_fissato.png


### 2.7 $I^{(\infty)}$ e $W$

Solo le due quantita' principali: lungo ciascuna linea, $W$ (nero) e $I^{(\infty)}$ (rosso), ciascuna divisa per il proprio massimo (scala log). Una figura per le linee a $\delta_{EI}$ fissato e una per quelle a $\delta_{IE}$ fissato. Sotto, la correlazione di rango di Spearman tra $W$ e $I^{(\infty)}$ linea per linea (solo punti con entrambe le quantita' finite).

In [10]:
# ============================================================
# 2.7 CONFRONTO TRA I^(inf) E W LUNGO LE LINEE
# ============================================================

def W_su_linea(res):
    # Valori di W nei punti di una linea
    return np.array([W[i, j] for (i, j) in res["punti"]])


def plot_I_inf_vs_W(results, targets, xlabel, legend_prefix, nome_file):
    # Un pannello per linea: W e I^(inf), ciascuna divisa per il proprio massimo lungo la linea
    n_cols = 3
    n_rows = int(np.ceil(len(targets) / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f"{legend_prefix} = {t:g}" for t in targets],
                        horizontal_spacing=0.07, vertical_spacing=0.14)
    serie = [("W", "W", "black", "solid"), ("I", "I<sup>(∞)</sup>", "crimson", "dash")]

    for idx, target in enumerate(targets):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        res = results[target]
        order = np.argsort(res["var_vals"])          # ordina i punti lungo la linea
        xv = res["var_vals"][order]
        valori = {"W": W_su_linea(res)[order], "I": res["I"][order]}

        for chiave, nome, colore, stile in serie:
            # scala log: solo valori finiti e positivi
            v = np.where(np.isfinite(valori[chiave]) & (valori[chiave] > 0), valori[chiave], np.nan)
            if np.all(np.isnan(v)):
                continue
            fig.add_trace(go.Scatter(
                x=xv, y=v / np.nanmax(v), mode="lines+markers",
                line=dict(color=colore, dash=stile, width=1.5), marker=dict(size=4, color=colore),
                name=nome, legendgroup=chiave, showlegend=(idx == 0),
                customdata=valori[chiave],             # valore non normalizzato, mostrato nell'hover
                hovertemplate=(f"{nome}<br>{xlabel}=%{{x:.3g}}<br>valore=%{{customdata:.4g}}"
                               "<br>(normalizzato %{y:.3g})<extra></extra>"),
            ), row=row, col=col)

        fig.update_xaxes(title_text=xlabel, row=row, col=col)
        fig.update_yaxes(title_text="valore / massimo", type="log", row=row, col=col)

    fig.update_layout(title=f"W e I<sup>(∞)</sup> lungo le linee a {legend_prefix} fissato (ciascuna / proprio massimo)",
                      height=420 * n_rows + 80, width=1500, **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_I_inf_vs_W(results_dei, valori_dei, "δ<sub>IE</sub>", "δ<sub>EI</sub>", "I_inf_vs_W_delta_EI")
plot_I_inf_vs_W(results_die, valori_die, "δ<sub>EI</sub>", "δ<sub>IE</sub>", "I_inf_vs_W_delta_IE")

# Correlazione di rango di Spearman tra W e I^(inf), linea per linea
rows = []
for nome, results in [("delta_EI", results_dei), ("delta_IE", results_die)]:
    for target, res in results.items():
        df = pd.DataFrame({"W": W_su_linea(res), "I_inf": res["I"]})
        df = df[np.isfinite(df).all(axis=1)]
        rows.append({
            "linea": f"{nome} = {target}",
            "n punti": len(df),
            "Spearman(W, I_inf)": df["W"].corr(df["I_inf"], method="spearman") if len(df) > 2 else np.nan,
        })
print("Correlazione di rango tra W e I_inf, linea per linea:")
display(pd.DataFrame(rows).round(3))

Figura salvata: figure_esportate_FDR_ottobre\I_inf_vs_W_delta_EI.png


Figura salvata: figure_esportate_FDR_ottobre\I_inf_vs_W_delta_IE.png


Correlazione di rango tra W e I_inf, linea per linea:


,linea,n punti,"Spearman(W, I_inf)"
0,delta_EI = -5,140,0.928
1,delta_EI = -2.5,140,0.914
2,delta_EI = -0.5,140,0.916
3,delta_EI = 0,140,0.954
4,delta_EI = 2.5,51,0.990
5,delta_EI = 5,39,0.999
6,delta_IE = -5,139,1.000
7,delta_IE = -2.5,107,0.939
8,delta_IE = -0.5,74,0.995
9,delta_IE = 0,69,0.914


---
## 3. Diagnostica dei picchi

La posizione dei massimi e' la stessa per $I^{(\infty)}$ e $I_T$ (differiscono per il fattore costante $T$). Per $\tilde I$ puo' essere diversa, quindi i suoi massimi hanno tabelle separate.

In [11]:
def tabella_picchi(results, axis, chiave="I"):
    # Per ogni linea: punto dove la quantita' 'chiave' e' massima e grandezze in quel punto.
    # (il parametro 'axis' non e' usato)
    # NOTA (da discutere): il massimo puo' cadere sul bordo della griglia
    # (es. linea delta_IE = -5, massimo a delta_EI = 7.0): li' non e' un vero picco.
    rows = []
    for target, res in results.items():
        I_vals = res[chiave]
        valid = np.isfinite(I_vals) & (I_vals > 0)
        if valid.sum() < 3:
            rows.append({"target": target, "n_validi": int(valid.sum()), "reason": "meno di 3 punti validi"})
            continue
        k = int(np.nanargmax(np.where(valid, I_vals, np.nan)))
        i, j = res["punti"][k]
        is_c = int(data["is_complex"][i, j])
        lam = data["lambda_re"][i, j] if is_c == 1 else max(data["lambda1"][i, j], data["lambda2"][i, j])
        rows.append({
            "target": target,
            "delta_EI_picco": float(data["delta_EI"][i]),
            "delta_IE_picco": float(data["delta_IE"][j]),
            "I_inf (ms)": float(res["I"][k]),
            "I_T": float(res["I_T"][k]),
            "I_tilde": float(res["I_tilde"][k]),
            "beta_eff (=y_inf)": float(res["y_inf"][k]),
            "w": float(data["A_w"][i, j]),
            "tau_slow": float(-1 / lam),
            "E0": float(data["E0"][i, j]), "I0": float(data["I0"][i, j]),
            "complesso": is_c,
            "W": float(W[i, j]),
        })
    return pd.DataFrame(rows)


# Massimi di I^(inf) (e quindi di I_T, che differisce solo per il fattore T)
T_peaks_dei = tabella_picchi(results_dei, "dei")
T_peaks_die = tabella_picchi(results_die, "die")
print("Massimi lungo le linee a delta_EI fissato:")
display(T_peaks_dei)
print("Massimi lungo le linee a delta_IE fissato:")
display(T_peaks_die)

# Massimi di I_tilde (possono cadere in punti diversi)
T_peaks_tilde_dei = tabella_picchi(results_dei, "dei", chiave="I_tilde")
T_peaks_tilde_die = tabella_picchi(results_die, "die", chiave="I_tilde")
print("Massimi di I_tilde = I_inf/tau_slow, linee a delta_EI fissato:")
display(T_peaks_tilde_dei)
print("Massimi di I_tilde = I_inf/tau_slow, linee a delta_IE fissato:")
display(T_peaks_tilde_die)

# Confronto delle posizioni dei massimi di I^(inf) e I_tilde
confronto = pd.DataFrame({
    "linea": [f"delta_EI = {t}" for t in T_peaks_dei["target"]] + [f"delta_IE = {t}" for t in T_peaks_die["target"]],
    "picco I_inf (dIE, dEI)": list(zip(T_peaks_dei["delta_IE_picco"], T_peaks_dei["delta_EI_picco"]))
                             + list(zip(T_peaks_die["delta_IE_picco"], T_peaks_die["delta_EI_picco"])),
    "picco I_tilde (dIE, dEI)": list(zip(T_peaks_tilde_dei["delta_IE_picco"], T_peaks_tilde_dei["delta_EI_picco"]))
                               + list(zip(T_peaks_tilde_die["delta_IE_picco"], T_peaks_tilde_die["delta_EI_picco"])),
})
confronto["stesso punto"] = [a == b for a, b in zip(confronto["picco I_inf (dIE, dEI)"], confronto["picco I_tilde (dIE, dEI)"])]
print("I massimi di I_inf e di I_tilde cadono nello stesso punto?")
display(confronto)

Massimi lungo le linee a delta_EI fissato:


,target,delta_EI_picco,delta_IE_picco,I_inf (ms),I_T,I_tilde,beta_eff (=y_inf),w,tau_slow,E0,I0,complesso,W
0,-5.0,-5.0,-1.6,0.154217,0.000005,0.169616,13.691784,-1.987187,0.909213,0.909084,0.678705,0,0.440624
1,-2.5,-2.5,-1.5,0.155945,0.000005,0.171028,13.748211,-1.938748,0.911808,0.908872,0.690061,0,0.425557
2,-0.5,-0.5,0.2,12.385333,0.000373,10.420318,0.695768,-0.077941,1.188575,0.865191,0.827066,1,1.148933
3,0.0,0.0,-0.4,119.504341,0.003599,113.316026,0.055089,-0.008345,1.054611,0.856422,0.768534,1,1.802818
4,2.5,2.5,-2.2,4.350404,0.000131,3.077423,-0.955403,0.239975,1.413651,0.836699,0.561404,0,3.137421
5,5.0,5.0,-3.3,3.032183,0.000091,2.229937,1.210098,-0.311463,1.359762,0.855751,0.444288,0,2.322791


Massimi lungo le linee a delta_IE fissato:


,target,delta_EI_picco,delta_IE_picco,I_inf (ms),I_T,I_tilde,beta_eff (=y_inf),w,tau_slow,E0,I0,complesso,W
0,-5.0,7.0,-5.0,0.124889,0.000004,0.136107,12.474234,-3.332746,0.917577,0.908479,0.253655,0,0.521507
1,-2.5,3.0,-2.5,8.591033,0.000259,6.311898,0.471146,-0.113855,1.361086,0.848581,0.534436,0,2.595241
2,-0.5,0.1,-0.5,102.334934,0.003082,99.194286,-0.061749,0.009801,1.031662,0.854695,0.757584,1,1.923190
3,0.0,-0.3,0.0,8.901552,0.000268,7.383143,-0.843513,0.117000,1.205659,0.854913,0.803839,1,1.711039
4,2.5,-1.1,2.5,9.393214,0.000283,3.680610,1.725346,-0.097082,2.552080,0.875389,0.906390,0,0.244249
5,5.0,-1.1,5.0,98.206895,0.002958,30.666494,0.180771,-0.010089,3.202417,0.872421,0.909054,0,0.263194


Massimi di I_tilde = I_inf/tau_slow, linee a delta_EI fissato:


,target,delta_EI_picco,delta_IE_picco,I_inf (ms),I_T,I_tilde,beta_eff (=y_inf),w,tau_slow,E0,I0,complesso,W
0,-5.0,-5.0,-1.6,0.154217,0.000005,0.169616,13.691784,-1.987187,0.909213,0.909084,0.678705,0,0.440624
1,-2.5,-2.5,-1.5,0.155945,0.000005,0.171028,13.748211,-1.938748,0.911808,0.908872,0.690061,0,0.425557
2,-0.5,-0.5,0.2,12.385333,0.000373,10.420318,0.695768,-0.077941,1.188575,0.865191,0.827066,1,1.148933
3,0.0,0.0,-0.4,119.504341,0.003599,113.316026,0.055089,-0.008345,1.054611,0.856422,0.768534,1,1.802818
4,2.5,2.5,-2.2,4.350404,0.000131,3.077423,-0.955403,0.239975,1.413651,0.836699,0.561404,0,3.137421
5,5.0,5.0,-3.3,3.032183,0.000091,2.229937,1.210098,-0.311463,1.359762,0.855751,0.444288,0,2.322791


Massimi di I_tilde = I_inf/tau_slow, linee a delta_IE fissato:


,target,delta_EI_picco,delta_IE_picco,I_inf (ms),I_T,I_tilde,beta_eff (=y_inf),w,tau_slow,E0,I0,complesso,W
0,-5.0,7.0,-5.0,0.124889,0.000004,0.136107,12.474234,-3.332746,0.917577,0.908479,0.253655,0,0.521507
1,-2.5,3.0,-2.5,8.591033,0.000259,6.311898,0.471146,-0.113855,1.361086,0.848581,0.534436,0,2.595241
2,-0.5,0.1,-0.5,102.334934,0.003082,99.194286,-0.061749,0.009801,1.031662,0.854695,0.757584,1,1.923190
3,0.0,-0.3,0.0,8.901552,0.000268,7.383143,-0.843513,0.117000,1.205659,0.854913,0.803839,1,1.711039
4,2.5,-1.1,2.5,9.393214,0.000283,3.680610,1.725346,-0.097082,2.552080,0.875389,0.906390,0,0.244249
5,5.0,-1.1,5.0,98.206895,0.002958,30.666494,0.180771,-0.010089,3.202417,0.872421,0.909054,0,0.263194


I massimi di I_inf e di I_tilde cadono nello stesso punto?


,linea,"picco I_inf (dIE, dEI)","picco I_tilde (dIE, dEI)",stesso punto
0,delta_EI = -5.0,"(-1.5999999999999996, -5.0)","(-1.5999999999999996, -5.0)",True
1,delta_EI = -2.5,"(-1.5, -2.5)","(-1.5, -2.5)",True
2,delta_EI = -0.5,"(0.20000000000000018, -0.5)","(0.20000000000000018, -0.5)",True
3,delta_EI = 0.0,"(-0.40000000000000036, 0.0)","(-0.40000000000000036, 0.0)",True
4,delta_EI = 2.5,"(-2.2, 2.5)","(-2.2, 2.5)",True
5,delta_EI = 5.0,"(-3.3000000000000003, 5.0)","(-3.3000000000000003, 5.0)",True
6,delta_IE = -5.0,"(-5.0, 7.0)","(-5.0, 7.0)",True
7,delta_IE = -2.5,"(-2.5, 3.0)","(-2.5, 3.0)",True
8,delta_IE = -0.5,"(-0.5, 0.10000000000000053)","(-0.5, 0.10000000000000053)",True
9,delta_IE = 0.0,"(0.0, -0.2999999999999998)","(0.0, -0.2999999999999998)",True


### 3.1 I picchi e il cambio di segno di $w$

$y_\infty = \beta_{eff} = -w/[(xw-yz)\sigma_{11}]$, quindi $y_\infty = 0$ dove $w = 0$. Poiche' $\Delta(t)$ contiene $y(t)/y_\infty$, vicino allo zero di $w$ l'integrando diventa grande per costruzione. Sulla griglia discreta lo zero di $w$ cade in generale **tra due punti**: il segnale e' il **cambio di segno** di $w$ (e di $\beta_{eff}$) tra punti adiacenti, non un valore di $w$ piccolo nel punto di griglia.

In [12]:
def plot_check_w(results, axis, targets, nome_file):
    # Per ogni linea: I^(inf) (asse sinistro, log) e w = A_ww (asse destro),
    # con la linea w = 0 tratteggiata, per vedere se i picchi stanno sugli zeri di w
    fig = make_subplots(
        rows=1, cols=len(targets),
        subplot_titles=[(f"δ<sub>EI</sub>={t}" if axis == "dei" else f"δ<sub>IE</sub>={t}") for t in targets],
        specs=[[{"secondary_y": True}] * len(targets)],
    )
    for col, target in enumerate(targets, start=1):
        res = results[target]
        order = np.argsort(res["var_vals"])
        xv = res["var_vals"][order]
        I_vals = np.where(np.isfinite(res["I"]) & (res["I"] > 0), res["I"], np.nan)[order]
        w_line = np.array([data["A_w"][i, j] for (i, j) in res["punti"]])[order]

        fig.add_trace(go.Scatter(x=xv, y=I_vals, mode="markers+lines",
                                 marker=dict(color="crimson", size=5), line=dict(color="crimson"),
                                 name="I<sup>(∞)</sup>", showlegend=(col == 1)),
                      row=1, col=col, secondary_y=False)
        fig.add_trace(go.Scatter(x=xv, y=w_line, mode="lines", line=dict(color="navy"),
                                 name="w", showlegend=(col == 1)),
                      row=1, col=col, secondary_y=True)
        fig.add_hline(y=0, line=dict(color="gray", dash="dash", width=1), row=1, col=col, secondary_y=True)
        fig.update_xaxes(title_text="δ<sub>IE</sub>" if axis == "dei" else "δ<sub>EI</sub>", row=1, col=col)
        fig.update_yaxes(title_text="I<sup>(∞)</sup> (ms)", type="log", row=1, col=col, secondary_y=False)
        fig.update_yaxes(title_text="w", row=1, col=col, secondary_y=True)
    fig.update_layout(title="Picco di I e zero di w", height=450, width=420 * len(targets) + 150,
                      **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_check_w(results_dei, "dei", [2.5, 5], "picchi_w_delta_EI")
plot_check_w(results_die, "die", [-0.5, 0, 2.5], "picchi_w_delta_IE")


# Verifica numerica: il picco cade accanto a un cambio di segno di w?
# (si confronta il segno di w nel picco con quello dei due punti vicini)
rows = []
for nome, results in [("delta_EI", results_dei), ("delta_IE", results_die)]:
    for target, res in results.items():
        valid = np.isfinite(res["I"]) & (res["I"] > 0)
        if valid.sum() < 3:
            continue
        order = np.argsort(res["var_vals"])
        I_o = np.where(valid, res["I"], np.nan)[order]
        w_o = np.array([data["A_w"][i, j] for (i, j) in res["punti"]])[order]
        k = int(np.nanargmax(I_o))
        vicini = [m for m in (k - 1, k + 1) if 0 <= m < len(w_o)]
        cambio = any(np.sign(w_o[m]) != np.sign(w_o[k]) for m in vicini)
        rows.append({"linea": f"{nome} = {target}", "picco a": float(res["var_vals"][order][k]),
                     "w nel picco": float(w_o[k]), "cambio di segno di w accanto al picco": cambio})
display(pd.DataFrame(rows))

Figura salvata: figure_esportate_FDR_ottobre\picchi_w_delta_EI.png


Figura salvata: figure_esportate_FDR_ottobre\picchi_w_delta_IE.png


,linea,picco a,w nel picco,cambio di segno di w accanto al picco
0,delta_EI = -5,-1.6,-1.987187,False
1,delta_EI = -2.5,-1.5,-1.938748,False
2,delta_EI = -0.5,0.2,-0.077941,True
3,delta_EI = 0,-0.4,-0.008345,True
4,delta_EI = 2.5,-2.2,0.239975,True
5,delta_EI = 5,-3.3,-0.311463,True
6,delta_IE = -5,7.0,-3.332746,False
7,delta_IE = -2.5,3.0,-0.113855,True
8,delta_IE = -0.5,0.1,0.009801,True
9,delta_IE = 0,-0.3,0.117000,True


---
## 4. Confronto con il tasso di produzione di entropia $W$

$W$ misura direttamente quanto il sistema e' lontano dall'equilibrio ($W = 0$ all'equilibrio). Se $I$ fosse una buona misura della stessa cosa, lungo una linea $I$ e $W$ dovrebbero crescere e calare insieme. Dove $I$ ha un picco e $W$ no, il picco viene dalla normalizzazione $y/y_\infty$ (zero di $\beta_{eff}$), non da un allontanamento dall'equilibrio.

Nei grafici ogni curva e' divisa per il proprio massimo lungo la linea, cosi' le tre quantita' (con unita' diverse) stanno sulla stessa scala. I punti con $W = \infty$ sono segnati da linee verticali grigie. La tabella riporta la correlazione di rango di Spearman tra $W$ e $I^{(\infty)}$, e tra $W$ e $\tilde I$ (solo punti con entrambe le quantita' finite): vicino a $+1$ crescono insieme, vicino a $0$ non sono legate.

### 4.1 $W$, $I^{(\infty)}$ e $\tilde I$

In [13]:
def W_lungo_linea(res):
    # Valori di W nei punti di una linea
    return np.array([W[i, j] for (i, j) in res["punti"]])


def plot_confronto_W(results, targets, xlabel, legend_prefix, nome_file):
    # Un pannello per linea: W, I^(inf), I_tilde, ciascuna divisa per il proprio
    # massimo lungo la linea (cosi' quantita' con unita' diverse stanno sulla stessa scala)
    n_cols = 3
    n_rows = int(np.ceil(len(targets) / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f"{legend_prefix} = {t:g}" for t in targets],
                        horizontal_spacing=0.07, vertical_spacing=0.14)
    serie = [("W", "black", "solid"), ("I", "crimson", "dash"), ("I_tilde", "seagreen", "dot")]
    nomi = {"W": "W", "I": "I<sup>(∞)</sup>", "I_tilde": "Ĩ"}

    for idx, target in enumerate(targets):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        res = results[target]
        order = np.argsort(res["var_vals"])
        xv = res["var_vals"][order]
        valori = {"W": W_lungo_linea(res)[order], "I": res["I"][order], "I_tilde": res["I_tilde"][order]}

        for chiave, colore, stile in serie:
            v = np.where(np.isfinite(valori[chiave]) & (valori[chiave] > 0), valori[chiave], np.nan)
            if np.all(np.isnan(v)):
                continue
            fig.add_trace(go.Scatter(
                x=xv, y=v / np.nanmax(v), mode="lines+markers",
                line=dict(color=colore, dash=stile, width=1.5), marker=dict(size=4, color=colore),
                name=nomi[chiave], legendgroup=chiave, showlegend=(idx == 0),
                customdata=valori[chiave],
                hovertemplate=(f"{nomi[chiave]}<br>{xlabel}=%{{x:.3g}}<br>valore=%{{customdata:.4g}}"
                               "<br>(normalizzato %{y:.3g})<extra></extra>"),
            ), row=row, col=col)

        # Linee verticali grigie nei punti con W = inf
        for xi in xv[np.isinf(valori["W"])]:
            fig.add_vline(x=xi, line=dict(color="lightgray", width=1), row=row, col=col)

        fig.update_xaxes(title_text=xlabel, row=row, col=col)
        fig.update_yaxes(title_text="valore / massimo", type="log", row=row, col=col)

    fig.update_layout(title=f"W, I<sup>(∞)</sup> e Ĩ lungo le linee a {legend_prefix} fissato (ciascuna / proprio massimo)",
                      height=420 * n_rows + 80, width=1500, **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_confronto_W(results_dei, valori_dei, "δ<sub>IE</sub>", "δ<sub>EI</sub>", "confronto_W_delta_EI")
plot_confronto_W(results_die, valori_die, "δ<sub>EI</sub>", "δ<sub>IE</sub>", "confronto_W_delta_IE")


# Correlazione di rango di Spearman tra W e I, linea per linea
# (solo punti dove W, I^(inf) e I_tilde sono tutti finiti).
# Spearman misura se due quantita' crescono e calano insieme (ordine dei valori),
# non se sono proporzionali.
rows = []
for nome, results in [("delta_EI", results_dei), ("delta_IE", results_die)]:
    for target, res in results.items():
        df = pd.DataFrame({"W": W_lungo_linea(res), "I": res["I"], "I_tilde": res["I_tilde"]})
        df = df[np.isfinite(df).all(axis=1)]
        rows.append({
            "linea": f"{nome} = {target}",
            "n punti": len(df),
            "Spearman(W, I_inf)": df["W"].corr(df["I"], method="spearman") if len(df) > 2 else np.nan,
            "Spearman(W, I_tilde)": df["W"].corr(df["I_tilde"], method="spearman") if len(df) > 2 else np.nan,
        })
print("Correlazione di rango tra W e le due versioni di I, linea per linea:")
display(pd.DataFrame(rows).round(3))

Figura salvata: figure_esportate_FDR_ottobre\confronto_W_delta_EI.png


Figura salvata: figure_esportate_FDR_ottobre\confronto_W_delta_IE.png


Correlazione di rango tra W e le due versioni di I, linea per linea:


,linea,n punti,"Spearman(W, I_inf)","Spearman(W, I_tilde)"
0,delta_EI = -5,140,0.928,0.928
1,delta_EI = -2.5,140,0.914,0.915
2,delta_EI = -0.5,140,0.916,0.608
3,delta_EI = 0,140,0.954,0.700
4,delta_EI = 2.5,51,0.990,0.880
5,delta_EI = 5,39,0.999,0.997
6,delta_IE = -5,139,1.000,0.981
7,delta_IE = -2.5,107,0.939,0.833
8,delta_IE = -0.5,74,0.995,0.918
9,delta_IE = 0,69,0.914,0.913


### 4.2 $I$ contro $W$ su tutto il piano (tutte le linee insieme)

La correlazione della tabella sopra e' calcolata **lungo ciascuna linea**. Per sostenere che $I$ e $W$ misurano la stessa cosa **in generale**, mettiamo insieme tutti i punti di tutte le linee in un unico grafico log-log di $I$ contro $W$:

- se i punti stanno su **una curva unica**, la relazione e' globale;
- se si dividono in **rami diversi** a seconda della linea, la relazione vale solo localmente.

Colori = linee. Con il simbolo **x** sono segnati i punti accanto a un cambio di segno di $w$ (zero di $\beta_{eff}$), dove $I$ ha i picchi dovuti alla normalizzazione. I punti all'incrocio di due linee compaiono una volta per linea nel grafico, ma una sola volta nella correlazione globale.

In [14]:
# Colori per le 12 linee
COLORI_LINEE = [
    "#1f77b4", "#aec7e8", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#000000",
]

# Tabella con tutti i punti di tutte le linee: W, I^(inf), I_tilde, I_T
# e un flag che dice se il punto e' accanto a un cambio di segno di w (zero di beta_eff)
record = []
for nome, results in [("δ<sub>EI</sub>", results_dei), ("δ<sub>IE</sub>", results_die)]:
    for target, res in results.items():
        order = np.argsort(res["var_vals"])
        w_o = np.array([data["A_w"][i, j] for (i, j) in res["punti"]])[order]
        # cambio[p] = True se w cambia segno tra il punto p e uno dei suoi vicini
        cambio = np.zeros(len(w_o), dtype=bool)
        cambio[:-1] |= np.sign(w_o[:-1]) != np.sign(w_o[1:])
        cambio[1:] |= np.sign(w_o[1:]) != np.sign(w_o[:-1])
        for pos, k in enumerate(order):
            i, j = res["punti"][k]
            record.append({
                "linea": f"{nome} = {target:g}", "i": i, "j": j,
                "W": W[i, j], "I": res["I"][k], "I_tilde": res["I_tilde"][k], "I_T": res["I_T"][k],
                "vicino_beta0": bool(cambio[pos]),
            })

df_all = pd.DataFrame(record)
# Solo punti con valori finiti e positivi (servono per la scala log)
df_ok = df_all[np.isfinite(df_all[["W", "I", "I_tilde"]]).all(axis=1)
               & (df_all["W"] > 0) & (df_all["I"] > 0)]

# Grafico log-log: I^(inf) contro W (sinistra) e I_tilde contro W (destra).
# Cerchi = punti normali, x = punti accanto a uno zero di beta_eff.
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                    subplot_titles=["I<sup>(∞)</sup> contro W", "Ĩ = I<sup>(∞)</sup>/τ<sub>slow</sub> contro W"])
for n, (linea, sub) in enumerate(df_ok.groupby("linea", sort=False)):
    colore = COLORI_LINEE[n % len(COLORI_LINEE)]
    for col, chiave in [(1, "I"), (2, "I_tilde")]:
        for flag, simbolo, dim in [(False, "circle", 6), (True, "x", 9)]:
            s = sub[sub["vicino_beta0"] == flag]
            if s.empty:
                continue
            fig.add_trace(go.Scattergl(
                x=s["W"], y=s[chiave], mode="markers",
                marker=dict(color=colore, symbol=simbolo, size=dim, opacity=0.8),
                name=linea, legendgroup=linea, showlegend=(col == 1 and not flag),
                hovertemplate=(f"{linea}<br>W=%{{x:.4g}}<br>{chiave}=%{{y:.4g}}"
                               + ("<br>accanto a zero di β<sub>eff</sub>" if flag else "")
                               + "<extra></extra>"),
            ), row=1, col=col)

fig.update_xaxes(title_text="W", type="log", row=1, col=1)
fig.update_xaxes(title_text="W", type="log", row=1, col=2)
fig.update_yaxes(title_text="I<sup>(∞)</sup> (ms)", type="log", row=1, col=1)
fig.update_yaxes(title_text="Ĩ", type="log", row=1, col=2)
fig.update_layout(title=f"I contro W, tutti i punti delle linee insieme — h = {h_label} (x = accanto a zero di β<sub>eff</sub>)",
                  height=600, width=1400, legend=dict(title="linea"), **ARTICLE_LAYOUT)
salva_figura(fig, "I_contro_W_globale")

# correlazione globale, ogni punto di griglia contato una sola volta
# (i punti all'incrocio di due linee compaiono due volte in df_ok)
uniq = df_ok.drop_duplicates(subset=["i", "j"])
uniq_senza = uniq[~uniq["vicino_beta0"]]
tab = pd.DataFrame({
    "insieme di punti": ["tutti", "senza punti accanto a zero di beta_eff"],
    "n punti": [len(uniq), len(uniq_senza)],
    "Spearman(W, I_inf)": [uniq["W"].corr(uniq["I"], method="spearman"),
                           uniq_senza["W"].corr(uniq_senza["I"], method="spearman")],
    "Spearman(W, I_tilde)": [uniq["W"].corr(uniq["I_tilde"], method="spearman"),
                             uniq_senza["W"].corr(uniq_senza["I_tilde"], method="spearman")],
})
print("Correlazione di rango GLOBALE (tutte le linee insieme, punti unici):")
display(tab.round(3))

Figura salvata: figure_esportate_FDR_ottobre\I_contro_W_globale.png


Correlazione di rango GLOBALE (tutte le linee insieme, punti unici):


,insieme di punti,n punti,"Spearman(W, I_inf)","Spearman(W, I_tilde)"
0,tutti,1150,0.868,0.793
1,senza punti accanto a zero di beta_eff,1129,0.869,0.800


---
## Appendice: le tre definizioni di $I$

Le tre definizioni della Sez. 2.3: $I^{(\infty)}$ (senza $1/T$), $I_T = I^{(\infty)}/T$ (con $1/T$ globale) e $\tilde I = I^{(\infty)}/\tau_{slow}$ (tempo locale).

### A.1 Le tre definizioni lungo le linee

Tre pannelli affiancati, uno per definizione. Sotto, la verifica che $I^{(\infty)}/I_T$ valga $T$ in tutti i punti.

In [15]:
# Colori e simboli delle 6 linee
PLOTLY_COLORS = [
    "rgb(31,119,180)", "rgb(174,199,232)", "rgb(214,39,40)",
    "rgb(44,160,44)", "rgb(255,127,14)", "rgb(148,103,189)",
]
PLOTLY_SYMBOLS = ["circle", "square", "triangle-up", "diamond", "pentagon", "star"]


# Tre pannelli affiancati: I^(inf), I_T, I_tilde (il nome dice "due" ma le definizioni sono tre)
def plot_I_due_definizioni(results, target_values, xlabel, legend_prefix, title, nome_file):
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Senza 1/T:  I<sup>(∞)</sup> = ∫ Δ dt  [ms]",
                        f"Con 1/T:  I<sub>T</sub> = I<sup>(∞)</sup>/T,  T = {T_glob:.0f} ms",
                        "Tempo locale:  Ĩ = I<sup>(∞)</sup>/τ<sub>slow</sub>"],
        horizontal_spacing=0.07,
    )
    for k, target in enumerate(target_values):
        res = results[target]
        order = np.argsort(res["var_vals"])        # ordina i punti lungo la linea
        xv = res["var_vals"][order]
        color = PLOTLY_COLORS[k % len(PLOTLY_COLORS)]
        symbol = PLOTLY_SYMBOLS[k % len(PLOTLY_SYMBOLS)]
        for col, chiave, nome_y in [(1, "I", "I_inf"), (2, "I_T", "I_T"), (3, "I_tilde", "I_tilde")]:
            yv = res[chiave][order]
            yv = np.where(np.isfinite(yv) & (yv > 0), yv, np.nan)   # scala log: solo valori > 0
            fig.add_trace(
                go.Scatter(
                    x=xv, y=yv, mode="lines+markers",
                    name=f"{legend_prefix} = {target:g}", legendgroup=f"t{target}",
                    showlegend=(col == 1),
                    line=dict(color=color, dash="dash", width=1.5),
                    marker=dict(symbol=symbol, size=6, color=color),
                    hovertemplate=(f"{legend_prefix} = {target:g}<br>{xlabel} = %{{x:.4g}}<br>"
                                   f"{nome_y} = %{{y:.6g}}<extra></extra>"),
                ),
                row=1, col=col,
            )
        for col in [1, 2, 3]:
            fig.update_xaxes(title_text=xlabel, row=1, col=col)
    fig.update_yaxes(title_text="I<sup>(∞)</sup> (ms)", type="log", row=1, col=1)
    fig.update_yaxes(title_text="I<sub>T</sub>", type="log", row=1, col=2)
    fig.update_yaxes(title_text="Ĩ", type="log", row=1, col=3)
    fig.update_layout(title=title, height=500, width=1600, hovermode="closest",
                      legend=dict(title=legend_prefix), **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_I_due_definizioni(results_dei, valori_dei, "δ<sub>IE</sub>", "δ<sub>EI</sub>",
                       f"Distanza dalla FDT, δ<sub>EI</sub> fissato — h = {h_label}",
                       "I_linee_delta_EI_fissato")
plot_I_due_definizioni(results_die, valori_die, "δ<sub>EI</sub>", "δ<sub>IE</sub>",
                       f"Distanza dalla FDT, δ<sub>IE</sub> fissato — h = {h_label}",
                       "I_linee_delta_IE_fissato")

# Verifica: I^(inf)/I_T deve valere T_glob in ogni punto
rapporti = np.concatenate([r["I"] / r["I_T"] for r in list(results_dei.values()) + list(results_die.values())])
rapporti = rapporti[np.isfinite(rapporti)]
print(f"Verifica: I_inf / I_T e' costante = T su tutti i punti? "
      f"min {rapporti.min():.1f}, max {rapporti.max():.1f} (T = {T_glob:.1f})")

Figura salvata: figure_esportate_FDR_ottobre\I_linee_delta_EI_fissato.png


Figura salvata: figure_esportate_FDR_ottobre\I_linee_delta_IE_fissato.png


Verifica: I_inf / I_T e' costante = T su tutti i punti? min 33202.6, max 33202.6 (T = 33202.6)


### A.2 Confronto con $W$

$W$ a confronto con le tre definizioni. Ogni curva e' divisa per il proprio massimo lungo la linea (scala log).

Nota: $I_T$ differisce da $I^{(\infty)}$ solo per il fattore costante $T$, quindi divisa per il proprio massimo coincide con $I^{(\infty)}$ (le due curve sono sovrapposte) e ha la stessa correlazione di Spearman. $\tilde I$ invece puo' avere un andamento diverso, perche' $\tau_{slow}$ cambia da punto a punto.

In [16]:
# ============================================================
# APPENDICE: W CONTRO I^(inf), I_T E I_tilde
# ============================================================

def plot_tre_definizioni_vs_W(results, targets, xlabel, legend_prefix, nome_file):
    # Un pannello per linea: W, I^(inf), I_T, I_tilde, ciascuna divisa per il proprio massimo
    n_cols = 3
    n_rows = int(np.ceil(len(targets) / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f"{legend_prefix} = {t:g}" for t in targets],
                        horizontal_spacing=0.07, vertical_spacing=0.14)
    # I_T e' disegnata a punti sopra I^(inf) (coincidono dopo la normalizzazione)
    serie = [("W", "W", "black", "solid", 1.5),
             ("I", "I<sup>(∞)</sup>", "crimson", "dash", 2.5),
             ("I_T", "I<sub>T</sub>", "gold", "dot", 1.5),
             ("I_tilde", "Ĩ", "seagreen", "dashdot", 1.5)]

    for idx, target in enumerate(targets):
        row, col = idx // n_cols + 1, idx % n_cols + 1
        res = results[target]
        order = np.argsort(res["var_vals"])
        xv = res["var_vals"][order]
        valori = {"W": W_su_linea(res)[order], "I": res["I"][order],
                  "I_T": res["I_T"][order], "I_tilde": res["I_tilde"][order]}

        for chiave, nome, colore, stile, spessore in serie:
            v = np.where(np.isfinite(valori[chiave]) & (valori[chiave] > 0), valori[chiave], np.nan)
            if np.all(np.isnan(v)):
                continue
            fig.add_trace(go.Scatter(
                x=xv, y=v / np.nanmax(v), mode="lines",
                line=dict(color=colore, dash=stile, width=spessore),
                name=nome, legendgroup=chiave, showlegend=(idx == 0),
                customdata=valori[chiave],
                hovertemplate=(f"{nome}<br>{xlabel}=%{{x:.3g}}<br>valore=%{{customdata:.4g}}"
                               "<br>(normalizzato %{y:.3g})<extra></extra>"),
            ), row=row, col=col)

        fig.update_xaxes(title_text=xlabel, row=row, col=col)
        fig.update_yaxes(title_text="valore / massimo", type="log", row=row, col=col)

    fig.update_layout(title=f"W e le tre definizioni di I lungo le linee a {legend_prefix} fissato",
                      height=420 * n_rows + 80, width=1500, **ARTICLE_LAYOUT)
    salva_figura(fig, nome_file)


plot_tre_definizioni_vs_W(results_dei, valori_dei, "δ<sub>IE</sub>", "δ<sub>EI</sub>", "appendice_tre_I_delta_EI")
plot_tre_definizioni_vs_W(results_die, valori_die, "δ<sub>EI</sub>", "δ<sub>IE</sub>", "appendice_tre_I_delta_IE")

# Spearman tra W e ciascuna delle tre definizioni, linea per linea
rows = []
for nome, results in [("delta_EI", results_dei), ("delta_IE", results_die)]:
    for target, res in results.items():
        df = pd.DataFrame({"W": W_su_linea(res), "I_inf": res["I"], "I_T": res["I_T"], "I_tilde": res["I_tilde"]})
        df = df[np.isfinite(df).all(axis=1)]
        riga = {"linea": f"{nome} = {target}", "n punti": len(df)}
        for c in ["I_inf", "I_T", "I_tilde"]:
            riga[f"Spearman(W, {c})"] = df["W"].corr(df[c], method="spearman") if len(df) > 2 else np.nan
        rows.append(riga)
print("Correlazione di rango tra W e le tre definizioni, linea per linea:")
display(pd.DataFrame(rows).round(3))

Figura salvata: figure_esportate_FDR_ottobre\appendice_tre_I_delta_EI.png


Figura salvata: figure_esportate_FDR_ottobre\appendice_tre_I_delta_IE.png


Correlazione di rango tra W e le tre definizioni, linea per linea:


,linea,n punti,"Spearman(W, I_inf)","Spearman(W, I_T)","Spearman(W, I_tilde)"
0,delta_EI = -5,140,0.928,0.928,0.928
1,delta_EI = -2.5,140,0.914,0.914,0.915
2,delta_EI = -0.5,140,0.916,0.916,0.608
3,delta_EI = 0,140,0.954,0.954,0.700
4,delta_EI = 2.5,51,0.990,0.990,0.880
5,delta_EI = 5,39,0.999,0.999,0.997
6,delta_IE = -5,139,1.000,1.000,0.981
7,delta_IE = -2.5,107,0.939,0.939,0.833
8,delta_IE = -0.5,74,0.995,0.995,0.918
9,delta_IE = 0,69,0.914,0.914,0.913
